In [4]:
%run datamodule_potsdam.py

torch.Size([16, 3, 256, 256, 1]) torch.Size([16, 1, 256, 256, 1])


In [233]:
batch["image"][tio.DATA].shape, batch["mask"][tio.DATA].shape

(torch.Size([16, 3, 256, 256, 1]), torch.Size([16, 1, 256, 256, 1]))

In [242]:
N = 6
images = batch["image"][tio.DATA][7:13].squeeze(-1)
masks = batch["mask"][tio.DATA][7:13].squeeze(-1)

images.shape, masks.shape

(torch.Size([6, 3, 256, 256]), torch.Size([6, 1, 256, 256]))

In [243]:
import torch
from torch.utils.tensorboard import SummaryWriter

writer = SummaryWriter("runs/min_experiment-v2")


def colorize_mask(mask):
    """
    mask: [H, W] with values in {0,1,2}
    returns: [3, H, W]
    """
    colors = torch.tensor(
        [
            [0, 0, 0],  # class 0 -> black
            [255, 0, 0],  # class 1 -> red
            [0, 255, 0],  # class 2 -> green
            [0, 0, 255],  # class 3 -> blue
            [255, 255, 0],  # class 4 -> yellow
            [255, 0, 255],  # class 5 -> magenta
        ],
        dtype=torch.uint8,
    )

    colored = colors[mask]  # [H, W, 3]
    return colored.permute(2, 0, 1)


In [244]:
colored_masks = torch.stack(
    [colorize_mask(m.long()) for m in masks.squeeze()]
)  # [N, 3, H, W]
colored_masks = colored_masks.float() / 255.0  # normalize to [0,1]

In [245]:
images.shape, colored_masks.shape

(torch.Size([6, 3, 256, 256]), torch.Size([6, 3, 256, 256]))

In [246]:
def overlay(image, mask, alpha=0.2):
    return image * (1 - alpha) + mask * alpha


overlays = overlay(images / 255, colored_masks)

In [247]:
overlays.min(), overlays.max(), overlays.shape

(tensor(0.0408), tensor(0.8149), torch.Size([6, 3, 256, 256]))

In [248]:
from torchvision.utils import make_grid

step = 0


# -----------------------
# Log
# -----------------------

grid = make_grid(overlays, nrow=N, padding=4)
writer.add_image("overlay", grid, step)
combined = torch.cat([images / 255, colored_masks], dim=0)  # [2N, 3, H, W]
grid = make_grid(combined, nrow=N, padding=4)  # N columns
writer.add_image("results", grid, 0)

writer.close()

print("✅ Logged! Now run:")
print("tensorboard --logdir=runs")

✅ Logged! Now run:
tensorboard --logdir=runs


In [249]:
images.shape, colored_masks.shape, overlays.shape

(torch.Size([6, 3, 256, 256]),
 torch.Size([6, 3, 256, 256]),
 torch.Size([6, 3, 256, 256]))